# C3 Picard + 50 Hz Notch + FD-F: Baseline Multi-Architecture Comparison

This notebook trains all five candidate neural architectures under identical
leave-one-subject-out (LOSO) conditions, producing the baseline results
reported for the dataset release:

1. Train all 5 models under identical, leakage-safe splits (same held-out
   subject per fold for every model).
2. Reset the random seed independently before every model/fold trains, so
   results do not depend on training order.
3. Report model parameter counts.
4. Produce the accuracy comparison across all 5 models and 3 calibration
   settings.
5. Produce the training and validation curve figure for a representative
   fold.


In [ ]:
from pathlib import Path
from IPython.display import clear_output, display
import time
import warnings
import sys
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from sklearn.metrics import confusion_matrix

warnings.filterwarnings("ignore")


for helper_dir in [
    Path.cwd(),
    Path.cwd() / 'python',
    Path.home() / 'Desktop' / 'Final Thesis'/ 'c3_picard_50HzNotch_FDF_package' / 'python',
]:
    if helper_dir.exists() and str(helper_dir) not in sys.path:
        sys.path.insert(0, str(helper_dir))

from deep_utils import (
    set_global_seed, get_device, ensure_dir,
    load_manifest, load_epoch_dataset,
    make_fold_indices, audit_split,
    fit_adaptive_zscore, apply_adaptive_zscore,
    build_model as _utils_build_model, train_model, evaluate_trial_level,
    append_row_csv, read_results, result_exists,
    summarize_results, save_best_by_calibration, cleanup_device,
    RESULT_COLUMNS, AUDIT_COLUMNS
)

class Config:
    epoch_root = Path.home() / 'Desktop' / 'Final Thesis' / 'c3_picard_50HzNotch_FDF_epochs'

    desktop_dir = Path.home() / 'Desktop'
    output_dir = desktop_dir / 'Final Thesis' / '5 model comparison results'

    seed = 42
    prefer_xpu = True

    max_epochs = 500
    patience = 30
    batch_size = 256
    base_lr = 1e-3
    weight_decay = 1e-4

    calib_lr = 1e-3
    calib_patience = 30
    calib_epochs_1trial = 180
    calib_epochs_5trials = 260

    pred_batch_size = 512

cfg = Config()
cfg.desktop_dir.mkdir(parents=True, exist_ok=True)
ensure_dir(cfg.output_dir)

set_global_seed(cfg.seed)
device = get_device(cfg.prefer_xpu)

print('Device:', device)
print('Epoch root:', cfg.epoch_root)
print('Output dir:', cfg.output_dir)

RESULT_FILE = cfg.output_dir / 'all_fold_results.csv'
HISTORY_FILE = cfg.output_dir / 'training_history.csv'
AUDIT_FILE = cfg.output_dir / 'split_leakage_audit.csv'
PRED_FILE = cfg.output_dir / 'fold_trial_predictions.csv'
SUMMARY_FILE = cfg.output_dir / 'summary.csv'
BEST_FILE = cfg.output_dir / 'best_by_calibration.csv'

FIG_ACCURACY_COMPARISON = cfg.output_dir / 'fig_01_model_accuracy_comparison.png'
FIG_SIGNIFICANCE_MATRIX = cfg.output_dir / 'fig_02_model_significance_matrix.png'
FIG_BEST_MODEL_DECLARATION = cfg.output_dir / 'fig_03_best_model_declaration.png'

FRIEDMAN_FILE = cfg.output_dir / 'friedman_summary.csv'
NEMENYI_FILE = cfg.output_dir / 'nemenyi_posthoc.csv'
NADEAU_BENGIO_FILE = cfg.output_dir / 'nadeau_bengio_pairwise.csv'
MODEL_MEAN_SUMMARY_FILE = cfg.output_dir / 'model_mean_accuracy_summary.csv'
PARAM_COUNT_FILE = cfg.output_dir / 'model_parameter_counts.csv'


In [ ]:

class CNNLSTMFamily(nn.Module):
    """
    CNN/LSTM family for EEG windows.

    Input shape: B x C x T
    C = EEG channels, T = time samples per window.
    """
    def __init__(
        self,
        n_channels: int,
        n_times: int,
        n_classes: int,
        n_lstm_layers: int = 1,
        bidirectional: bool = False,
        use_cnn: bool = True,
        conv_filters: int = 64,
        hidden: int = 64,
        dropout: float = 0.35,
    ):
        super().__init__()
        self.use_cnn = use_cnn
        self.bidirectional = bidirectional
        self.n_lstm_layers = n_lstm_layers

        if use_cnn:
            k = min(7, max(3, n_times if n_times % 2 == 1 else n_times - 1))
            self.conv = nn.Sequential(
                nn.Conv1d(n_channels, conv_filters, kernel_size=k, padding=k // 2, bias=False),
                nn.BatchNorm1d(conv_filters),
                nn.ELU(),
                nn.MaxPool1d(kernel_size=2) if n_times >= 4 else nn.Identity(),
                nn.Dropout(dropout),
            )
            lstm_input = conv_filters
        else:
            self.conv = None
            lstm_input = n_channels

        self.lstm = nn.LSTM(
            input_size=lstm_input,
            hidden_size=hidden,
            num_layers=n_lstm_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if n_lstm_layers > 1 else 0.0,
        )

        out_dim = hidden * (2 if bidirectional else 1)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(out_dim, n_classes)
        )

    def forward(self, x):
        if self.use_cnn:
            z = self.conv(x)
            z = z.transpose(1, 2)
        else:
            z = x.transpose(1, 2)

        out, _ = self.lstm(z)
        last = out[:, -1, :]
        return self.head(last)


MODEL_NAMES = [
    'CNN-1-LSTM',
    'CNN-2-LSTM',
    'CNN-2-Bi-LSTM',
    'CNN-3-LSTM',
    '3-LSTM',
]

MODEL_COLORS = {
    'CNN-1-LSTM': '#4C78A8',
    'CNN-2-LSTM': '#F58518',
    'CNN-2-Bi-LSTM': '#54A24B',
    'CNN-3-LSTM': '#B279A2',
    '3-LSTM': '#E45756',
}


def build_model(name: str, n_channels: int, n_times: int, n_classes: int) -> nn.Module:
    """
    Local model factory. This overrides the imported utility build_model to ensure all five
    required model names are available in this notebook.
    """
    name = str(name)

    if name == 'CNN-1-LSTM':
        return CNNLSTMFamily(n_channels, n_times, n_classes, n_lstm_layers=1, bidirectional=False, use_cnn=True)

    if name == 'CNN-2-LSTM':
        return CNNLSTMFamily(n_channels, n_times, n_classes, n_lstm_layers=2, bidirectional=False, use_cnn=True)

    if name == 'CNN-2-Bi-LSTM':
        return CNNLSTMFamily(n_channels, n_times, n_classes, n_lstm_layers=2, bidirectional=True, use_cnn=True)

    if name == 'CNN-3-LSTM':
        return CNNLSTMFamily(n_channels, n_times, n_classes, n_lstm_layers=3, bidirectional=False, use_cnn=True)

    if name == '3-LSTM':
        return CNNLSTMFamily(n_channels, n_times, n_classes, n_lstm_layers=3, bidirectional=False, use_cnn=False)

    return _utils_build_model(name, n_channels, n_times, n_classes)


In [ ]:
manifest = load_manifest(cfg.epoch_root)
manifest = manifest[manifest["mode"].astype(str).str.contains("win8")]
manifest = manifest[manifest["variant"].astype(str).str.contains("FDF")]
manifest = manifest[~manifest["variant"].astype(str).str.contains("noFDF")]
if len(manifest) == 0:
    raise RuntimeError(
        "No FD-F win8_0 rows found in the manifest after filtering. Check that "
        "epoch_root points at the right folder and that the FD-F export ran."
    )
display(manifest)

CALIBRATIONS = [
    ('pure_loso', 0),
    ('calib_approx5pct_1trial', 1),
    ('calib_20pct_5trials', 5),
]
CALIB_DISPLAY_NAMES = {
    'pure_loso': 'Pure LOSO',
    'calib_approx5pct_1trial': '1-trial calibration',
    'calib_20pct_5trials': '5-trial calibration',
}
CALIB_ORDER = [c[0] for c in CALIBRATIONS]

plan = []
for _, row in manifest.iterrows():
    for model_name in MODEL_NAMES:
        for calib_name, trials_per_class in CALIBRATIONS:
            plan.append(dict(
                row=row.to_dict(),
                variant=str(row['variant']),
                mode=str(row['mode']),
                model=model_name,
                calibration_name=calib_name,
                calib_trials_per_class=trials_per_class,
            ))

_first_ds_for_count = load_epoch_dataset(pd.Series(plan[0]['row']))
N_SUBJECTS = len(sorted(np.unique(_first_ds_for_count.participants).tolist()))
EXPECTED_FOLDS = len(plan) * N_SUBJECTS
N_CHANNELS_GLOBAL = _first_ds_for_count.X.shape[1]
N_TIMES_GLOBAL = _first_ds_for_count.X.shape[2]
N_CLASSES_GLOBAL = len(_first_ds_for_count.class_names)
print('Models:', MODEL_NAMES)
print('Calibration settings:', CALIB_ORDER)
print('Number of subjects detected:', N_SUBJECTS)
print('Plan items before subjects:', len(plan))
print('Expected total LOSO folds:', EXPECTED_FOLDS)
print('n_channels:', N_CHANNELS_GLOBAL, ' n_times:', N_TIMES_GLOBAL, ' n_classes:', N_CLASSES_GLOBAL)
del _first_ds_for_count


In [ ]:
def read_df(path, columns=None):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame(columns=columns)
    try:
        return pd.read_csv(path)
    except Exception as e:
        print('Could not read', path, e)
        return pd.DataFrame(columns=columns)

def show_progress(current=''):
    done = read_df(RESULT_FILE, RESULT_COLUMNS)
    n_done = len(done)
    total = EXPECTED_FOLDS if 'EXPECTED_FOLDS' in globals() else max(1, n_done)
    pct = 100.0 * n_done / max(1, total)
    bar_len = 40
    filled = int(round(bar_len * pct / 100))
    bar = '█' * filled + '░' * (bar_len - filled)
    print(f'Overall progress: {n_done}/{total} folds = {pct:.2f}%')
    print(bar)
    if current:
        print('Current:', current)
    if len(done):
        tmp = done.groupby(['variant','mode','model','calibration_name'], dropna=False).agg(
            n_folds=('subject','nunique'),
            mean_balanced_accuracy=('balanced_accuracy','mean'),
            mean_accuracy=('accuracy','mean'),
            mean_macro_f1=('macro_f1','mean'),
        ).reset_index().sort_values('mean_balanced_accuracy', ascending=False)
        display(tmp.head(25))


In [ ]:

for plan_i, item in enumerate(plan, 1):
    row = pd.Series(item['row'])
    mode = item['mode']
    variant = item['variant']
    model_name = item['model']
    calib_name = item['calibration_name']
    calib_trials_per_class = int(item['calib_trials_per_class'])

    ds = load_epoch_dataset(row)
    subjects = sorted(np.unique(ds.participants).tolist())
    n_classes = len(ds.class_names)
    n_channels = ds.X.shape[1]
    n_times = ds.X.shape[2]
    window_sec = ds.window_sec

    for heldout in subjects:
        if result_exists(RESULT_FILE, mode, model_name, calib_name, heldout):
            continue

        clear_output(wait=True)
        show_progress(
            f'[{plan_i}/{len(plan)}] {variant} | {mode} | {model_name} | '
            f'{calib_name} | heldout={heldout} | train separately'
        )
        start_total = time.time()

        split = make_fold_indices(ds, heldout, calib_trials_per_class, seed=cfg.seed)
        audit = audit_split(ds, heldout, split, calib_name, model_name)
        append_row_csv(AUDIT_FILE, audit, AUDIT_COLUMNS)

        train_idx = split['train_idx']
        val_idx = split['val_idx']
        calib_idx = split['calib_idx']
        test_idx = split['test_idx']

        mean, std = fit_adaptive_zscore(ds.X[train_idx])
        X_train = apply_adaptive_zscore(ds.X[train_idx], mean, std)
        y_train = ds.y[train_idx]
        X_val = apply_adaptive_zscore(ds.X[val_idx], mean, std) if len(val_idx) else None
        y_val = ds.y[val_idx] if len(val_idx) else None
        X_test = apply_adaptive_zscore(ds.X[test_idx], mean, std)
        y_test = ds.y[test_idx]
        tu_test = ds.trial_uid[test_idx]

        set_global_seed(cfg.seed)

        model = build_model(model_name, n_channels, n_times, n_classes)
        model, info = train_model(
            model, X_train, y_train, X_val, y_val, device,
            max_epochs=cfg.max_epochs, patience=cfg.patience, batch_size=cfg.batch_size,
            lr=cfg.base_lr, weight_decay=cfg.weight_decay, verbose=False,
        )
        train_runtime = float(info.get('runtime_sec', 0.0))
        calib_runtime = 0.0

        for h in info.get('history', []):
            append_row_csv(HISTORY_FILE, dict(
                variant=variant, mode=mode, model=model_name, calibration_name=calib_name,
                subject=heldout, phase='source', epoch=h['epoch'], train_loss=h['train_loss'],
                val_accuracy=h.get('val_accuracy', ''), val_balanced_accuracy=h.get('val_balanced_accuracy', ''),
                val_macro_f1=h.get('val_macro_f1', ''),
            ), columns=['variant','mode','model','calibration_name','subject','phase','epoch',
                       'train_loss','val_accuracy','val_balanced_accuracy','val_macro_f1'])

        if len(calib_idx) > 0:
            X_calib = apply_adaptive_zscore(ds.X[calib_idx], mean, std)
            y_calib = ds.y[calib_idx]
            calib_epochs = cfg.calib_epochs_5trials if calib_trials_per_class >= 5 else cfg.calib_epochs_1trial
            model, cinfo = train_model(
                model, X_calib, y_calib, X_val, y_val, device,
                max_epochs=calib_epochs, patience=cfg.calib_patience, batch_size=min(cfg.batch_size, 256),
                lr=cfg.calib_lr, weight_decay=cfg.weight_decay, verbose=False,
            )
            calib_runtime = float(cinfo.get('runtime_sec', 0.0))
            for h in cinfo.get('history', []):
                append_row_csv(HISTORY_FILE, dict(
                    variant=variant, mode=mode, model=model_name, calibration_name=calib_name,
                    subject=heldout, phase='calibration', epoch=h['epoch'], train_loss=h['train_loss'],
                    val_accuracy=h.get('val_accuracy', ''), val_balanced_accuracy=h.get('val_balanced_accuracy', ''),
                    val_macro_f1=h.get('val_macro_f1', ''),
                ), columns=['variant','mode','model','calibration_name','subject','phase','epoch',
                           'train_loss','val_accuracy','val_balanced_accuracy','val_macro_f1'])

        metrics, pred_df = evaluate_trial_level(model, X_test, y_test, tu_test, device, batch_size=cfg.pred_batch_size)
        total_runtime = time.time() - start_total

        result_row = dict(
            pipeline='C3_Picard_50HzNotch_FDF_MultiModel',
            variant=variant,
            mode=mode,
            window_sec=window_sec,
            model=model_name,
            calibration_name=calib_name,
            calib_trials_per_class=calib_trials_per_class,
            subject=heldout,
            n_train_subjects=len(split['train_subjects']),
            n_val_subjects=len(split['val_subjects']),
            n_train_windows=len(train_idx),
            n_val_windows=len(val_idx),
            n_calib_trials=audit['n_calib_trials'],
            n_calib_windows=len(calib_idx),
            n_test_trials=audit['n_test_trials'],
            n_test_windows=len(test_idx),
            accuracy=metrics['accuracy'],
            balanced_accuracy=metrics['balanced_accuracy'],
            macro_f1=metrics['macro_f1'],
            best_epoch=info.get('best_epoch', ''),
            epochs_run=info.get('epochs_run', ''),
            train_runtime_sec=train_runtime,
            calib_runtime_sec=calib_runtime,
            total_runtime_sec=total_runtime,
            device=str(device),
            seed=cfg.seed,
        )
        append_row_csv(RESULT_FILE, result_row, RESULT_COLUMNS)

        if len(pred_df):
            pred_df = pred_df.copy()
            pred_df['pipeline'] = 'C3_Picard_50HzNotch_FDF_MultiModel'
            pred_df['variant'] = variant
            pred_df['mode'] = mode
            pred_df['model'] = model_name
            pred_df['calibration_name'] = calib_name
            pred_df['subject'] = heldout
            pred_df['label_true'] = pred_df['y_true'].map(lambda i: ds.class_names[int(i)] if int(i) < len(ds.class_names) else str(i))
            pred_df['label_pred'] = pred_df['y_pred'].map(lambda i: ds.class_names[int(i)] if int(i) < len(ds.class_names) else str(i))
            header = not Path(PRED_FILE).exists()
            pred_df.to_csv(PRED_FILE, mode='a', header=header, index=False)

        cleanup_device(device)

clear_output(wait=True)
show_progress('C3 Picard + 50 Hz Notch + FD-F multi-model LOSO run completed')
summary = summarize_results(RESULT_FILE, SUMMARY_FILE)
best = save_best_by_calibration(summary, BEST_FILE)
print('Saved:', SUMMARY_FILE)
print('Saved:', BEST_FILE)
display(best)


## Model complexity

Real parameter counts for each architecture, computed directly from the actual dataset dimensions
(not estimated). These are used later as one of the tie-breaking criteria if multiple models turn
out to be statistically indistinguishable.


In [ ]:
param_counts = {}
for model_name in MODEL_NAMES:
    m = build_model(model_name, N_CHANNELS_GLOBAL, N_TIMES_GLOBAL, N_CLASSES_GLOBAL)
    param_counts[model_name] = sum(p.numel() for p in m.parameters())
    del m

param_df = pd.DataFrame(sorted(param_counts.items(), key=lambda kv: kv[1]),
                         columns=['model', 'n_parameters'])
param_df.to_csv(PARAM_COUNT_FILE, index=False)
display(param_df)


## Accuracy comparison across the 5 models

Mean balanced accuracy +/- SD across the 20 LOSO folds, for each model at each calibration setting.
This is descriptive only - do not rank models by this chart alone; the statistical tests below
determine which differences are real.


In [ ]:
import matplotlib.ticker as mtick

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300, "font.family": "DejaVu Sans", "font.size": 11,
    "axes.titlesize": 14, "axes.titleweight": "bold", "axes.labelsize": 12,
    "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.fontsize": 9.5,
    "axes.spines.top": False, "axes.spines.right": False,
})

results_df = pd.read_csv(RESULT_FILE)

acc_summary = results_df.groupby(['model', 'calibration_name'])['balanced_accuracy'].agg(['mean', 'std']).reset_index()

fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(CALIB_ORDER))
width = 0.15

for i, model_name in enumerate(MODEL_NAMES):
    means, stds = [], []
    for calib in CALIB_ORDER:
        row = acc_summary[(acc_summary['model'] == model_name) & (acc_summary['calibration_name'] == calib)]
        means.append(float(row['mean'].values[0]) if len(row) else 0.0)
        stds.append(float(row['std'].values[0]) if len(row) else 0.0)
    offset = (i - (len(MODEL_NAMES) - 1) / 2) * width
    bars = ax.bar(x + offset, means, width, yerr=stds, capsize=2.5, label=model_name,
                   color=MODEL_COLORS[model_name], edgecolor='white', linewidth=0.4,
                   error_kw=dict(elinewidth=0.9, ecolor='#333333'))

ax.set_xticks(x)
ax.set_xticklabels([CALIB_DISPLAY_NAMES[c] for c in CALIB_ORDER])
ax.set_ylabel('Balanced accuracy')
ax.set_title('Model accuracy comparison across calibration settings\n(mean ± SD across 20 LOSO folds)')
ax.legend(loc='upper left', ncol=5, frameon=False, bbox_to_anchor=(0, 1.16))
ax.set_ylim(0, 1.05)

fig.tight_layout()
fig.savefig(FIG_ACCURACY_COMPARISON, dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(str(FIG_ACCURACY_COMPARISON).replace('.png', '.pdf'), bbox_inches='tight', facecolor='white')
plt.show()
print('Saved:', FIG_ACCURACY_COMPARISON)


In [ ]:

FIG_TRAINING_CURVES = cfg.output_dir / 'fig_03_training_curves.png'

from matplotlib.lines import Line2D

COLOR_LOSS = '#D55E00'
COLOR_ACC = '#0072B2'
TARGET_MODEL = 'CNN-2-Bi-LSTM'

history = pd.read_csv(HISTORY_FILE)
results = pd.read_csv(RESULT_FILE)

history = history[history['model'] == TARGET_MODEL]
results = results[results['model'] == TARGET_MODEL]

if len(history) == 0 or len(results) == 0:
    print(f'No history/results found yet for {TARGET_MODEL} - run the training loop first.')
else:
    rank_dev = None
    for calib in CALIB_ORDER:
        sub = results[results['calibration_name'] == calib][['subject', 'balanced_accuracy']].copy()
        n = len(sub)
        sub['rank'] = sub['balanced_accuracy'].rank(method='average')
        median_rank = (n + 1) / 2
        sub[f'dev_{calib}'] = (sub['rank'] - median_rank).abs()
        sub = sub[['subject', f'dev_{calib}']].set_index('subject')
        rank_dev = sub if rank_dev is None else rank_dev.join(sub, how='outer')

    rank_dev['total_dev'] = rank_dev.sum(axis=1)
    representative_subject = rank_dev['total_dev'].idxmin()
    print('Rank-deviation table (lower total_dev = more consistently typical across all 3 settings):')
    print(rank_dev.sort_values('total_dev').to_string())
    print()
    print(f'Single representative subject chosen for all 3 panels: {representative_subject}')

    fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))

    for i, calib in enumerate(CALIB_ORDER):
        ax = axes[i]
        subj = representative_subject
        fold_hist = history[(history['calibration_name'] == calib) & (history['subject'] == subj)]
        source_hist = fold_hist[fold_hist['phase'] == 'source'].sort_values('epoch')
        calib_hist = fold_hist[fold_hist['phase'] == 'calibration'].sort_values('epoch')

        fold_result = results[(results['calibration_name'] == calib) & (results['subject'] == subj)].iloc[0]
        source_best_epoch = int(fold_result['best_epoch'])

        source_end = source_hist['epoch'].max()
        if len(calib_hist):
            calib_hist = calib_hist.copy()
            calib_hist['epoch_plot'] = calib_hist['epoch'] + source_end
            calib_best_epoch = int(calib_hist.loc[calib_hist['val_balanced_accuracy'].idxmax(), 'epoch']) + source_end
        else:
            calib_best_epoch = None

        ax2 = ax.twinx()

        ax.plot(source_hist['epoch'], source_hist['train_loss'], color=COLOR_LOSS, linewidth=1.6)
        ax2.plot(source_hist['epoch'], source_hist['val_balanced_accuracy'], color=COLOR_ACC, linewidth=1.6)
        if len(calib_hist):
            ax.plot(calib_hist['epoch_plot'], calib_hist['train_loss'], color=COLOR_LOSS, linewidth=1.6, linestyle='--')
            ax2.plot(calib_hist['epoch_plot'], calib_hist['val_balanced_accuracy'], color=COLOR_ACC, linewidth=1.6, linestyle='--')
            ax.axvline(source_end, color='#999999', linestyle=':', linewidth=1.2, zorder=1)
            ax.text(source_end, ax.get_ylim()[1], '  calibration\n  fine-tuning starts', fontsize=7,
                    color='#777777', va='top', ha='left')

        ax.axvline(source_best_epoch, color='#222222', linestyle='--', linewidth=1.3, zorder=1)
        if calib_best_epoch is not None:
            ax.axvline(calib_best_epoch, color='#222222', linestyle='--', linewidth=1.3, zorder=1)

        ax.set_xlabel('Epoch')
        ax.set_ylabel('Train loss', color=COLOR_LOSS)
        ax2.set_ylabel('Val. balanced accuracy', color=COLOR_ACC)
        ax.tick_params(axis='y', labelcolor=COLOR_LOSS)
        ax2.tick_params(axis='y', labelcolor=COLOR_ACC)
        ax.set_title(f'{CALIB_DISPLAY_NAMES[calib]}', fontsize=10.5, fontweight='bold')

    legend_handles = [
        Line2D([0], [0], color=COLOR_LOSS, lw=2, label='Train loss'),
        Line2D([0], [0], color=COLOR_ACC, lw=2, label='Val. balanced accuracy'),
        Line2D([0], [0], color='#222222', lw=1.3, linestyle='--', label='Early-stopping epoch'),
    ]
    fig.legend(handles=legend_handles, loc='upper center', ncol=3, fontsize=9.5,
               bbox_to_anchor=(0.5, 1.06), frameon=False)
    fig.suptitle(f'{TARGET_MODEL} training and validation curves \u2014 subject {representative_subject}, '
                 f'representative across all calibration settings',
                 fontsize=12.5, fontweight='bold', y=1.14)

    fig.tight_layout()
    fig.savefig(FIG_TRAINING_CURVES, dpi=300, bbox_inches='tight', facecolor='white')
    fig.savefig(str(FIG_TRAINING_CURVES).replace('.png', '.pdf'), bbox_inches='tight', facecolor='white')
    plt.show()
    print('Saved:', FIG_TRAINING_CURVES)